# Export DimDate to OneDrive
Reads the DimDate table and uploads it as a CSV file to OneDrive (sagar_bathe@yahoo.com) using Microsoft Graph API with device code authentication.

In [1]:
# Install MSAL for authentication
%pip install msal

StatementMeta(, e330ca61-da1f-4b5c-a6cd-e830323cfc2e, 8, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
import msal
import requests
import os
import tempfile

# OneDrive login: sagar_bathe@yahoo.com
# Using device code flow for personal Microsoft accounts

# Microsoft Graph public client app (multi-tenant + personal accounts)
CLIENT_ID = "9ee8588e-ff74-4d6e-90fc-475866a67c6b"
TENANT = "consumers"  # 'consumers' for personal Microsoft accounts like @yahoo.com

authority = f"https://login.microsoftonline.com/{TENANT}"
scopes = ["Files.ReadWrite", "User.Read"]

# Create public client app
app = msal.PublicClientApplication(
    client_id=CLIENT_ID,
    authority=authority
)

# Initiate device code flow
flow = app.initiate_device_flow(scopes=scopes)
if "user_code" not in flow:
    raise Exception(f"Failed to create device flow: {flow.get('error_description', 'Unknown error')}")

print(f"\n{'='*60}")
print(f"To sign in, open a browser and go to:")
print(f"  {flow['verification_uri']}")
print(f"\nEnter this code: {flow['user_code']}")
print(f"Login with: sagar_bathe@yahoo.com")
print(f"{'='*60}\n")
print("Waiting for authentication...")

StatementMeta(, e330ca61-da1f-4b5c-a6cd-e830323cfc2e, 10, Finished, Available, Finished, False)


To sign in, open a browser and go to:
  https://www.microsoft.com/link

Enter this code: ZU6Z8SSK
Login with: sagar_bathe@yahoo.com

Waiting for authentication...


In [3]:
# Complete authentication
result = app.acquire_token_by_device_flow(flow)

if "access_token" in result:
    access_token = result["access_token"]
    print(f"Authenticated as: {result.get('id_token_claims', {}).get('preferred_username', 'OK')}")
else:
    raise Exception(f"Authentication failed: {result.get('error_description', result.get('error', 'Unknown'))}")

StatementMeta(, e330ca61-da1f-4b5c-a6cd-e830323cfc2e, 11, Finished, Available, Finished, False)

Authenticated as: sagar_bathe@yahoo.com


In [4]:
# Read DimDate table
df_dimdate = spark.sql("SELECT * FROM DimDate")
print(f"DimDate rows: {df_dimdate.count()}")
display(df_dimdate.limit(5))

StatementMeta(, e330ca61-da1f-4b5c-a6cd-e830323cfc2e, 12, Finished, Available, Finished, False)

DimDate rows: 3652


SynapseWidget(Synapse.DataFrame, 5b148e2e-87af-47e9-9fa6-26f5fe3e7233)

In [5]:
# Convert to CSV
pdf = df_dimdate.toPandas()
temp_path = os.path.join(tempfile.gettempdir(), "DimDate.csv")
pdf.to_csv(temp_path, index=False)
file_size = os.path.getsize(temp_path)
print(f"CSV saved locally: {temp_path} ({file_size:,} bytes)")

StatementMeta(, e330ca61-da1f-4b5c-a6cd-e830323cfc2e, 13, Finished, Available, Finished, False)

CSV saved locally: /tmp/DimDate.csv (355,754 bytes)


In [6]:
# Upload to OneDrive root folder
# Using Microsoft Graph simple upload (for files < 4MB)
# For larger files, use upload session

graph_headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "text/csv"
}

upload_filename = "DimDate.csv"
upload_url = f"https://graph.microsoft.com/v1.0/me/drive/root:/{upload_filename}:/content"

with open(temp_path, "rb") as f:
    file_content = f.read()

if file_size < 4 * 1024 * 1024:  # Simple upload for < 4MB
    resp = requests.put(upload_url, headers=graph_headers, data=file_content)
else:
    # Create upload session for large files
    session_url = f"https://graph.microsoft.com/v1.0/me/drive/root:/{upload_filename}:/createUploadSession"
    session_resp = requests.post(session_url, headers={"Authorization": f"Bearer {access_token}"}, json={"item": {"name": upload_filename}})
    upload_session = session_resp.json()
    upload_endpoint = upload_session["uploadUrl"]
    chunk_size = 10 * 1024 * 1024
    for i in range(0, file_size, chunk_size):
        chunk = file_content[i:i+chunk_size]
        end = min(i + chunk_size, file_size) - 1
        chunk_headers = {
            "Content-Range": f"bytes {i}-{end}/{file_size}",
            "Content-Length": str(len(chunk))
        }
        resp = requests.put(upload_endpoint, headers=chunk_headers, data=chunk)

if resp.status_code in [200, 201]:
    file_info = resp.json()
    print(f"\nFile uploaded to OneDrive successfully!")
    print(f"  Name: {file_info.get('name')}")
    print(f"  Size: {file_info.get('size'):,} bytes")
    print(f"  Web URL: {file_info.get('webUrl', 'N/A')}")
else:
    print(f"Upload failed: {resp.status_code}")
    print(resp.text)

StatementMeta(, e330ca61-da1f-4b5c-a6cd-e830323cfc2e, 14, Finished, Available, Finished, False)


File uploaded to OneDrive successfully!
  Name: DimDate.csv
  Size: 355,754 bytes
  Web URL: https://onedrive.live.com/personal/9019314c90087f1a/_layouts/15/doc.aspx?resid=6a5389ef-293a-4fda-bda2-e73e37b7ab6c&cid=9019314c90087f1a


In [7]:
# Cleanup
os.remove(temp_path)
print("Temp file cleaned up. Done!")

StatementMeta(, e330ca61-da1f-4b5c-a6cd-e830323cfc2e, 15, Finished, Available, Finished, False)

Temp file cleaned up. Done!
